# Semptom-İlaç İlişkisi Bulgusu

Bilinen psikiyatrik semptom kategorileri ile bu kategorilere karşılık gelen
ziyaretlerde en sık kullanılan ilaçlar arasındaki ilişkiyi, veri setinin
tamamı üzerinden sistematik olarak analiz eder.

**Yöntem:** Her kategori, örnek Türkçe ifadelerle (tohum) temsil edilir. Her
tohum ayrı ayrı `find_similar_phrases` ile tüm semptom kayıtlarına karşı
taranır (ortalama vektör KULLANILMAZ — bkz. tasarım dokümanı, ortalama
almanın gerçek verilerde eşleşmeleri sulandırdığı doğrulandı). Bir kayıt
herhangi bir tohuma eşik üstü benzerse, o kaydın ziyareti kategoriye dahil
edilir.

**Sınırlama:** Bu tanımlayıcı/istatistiksel bir bulgudur — "bu semptom
grubunda bu ilaçlar sık kullanılmış" der, nedensellik ("bu ilaç bu semptomu
tedavi eder") iddia etmez.

In [1]:
import sqlite3
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer

sys.path.insert(0, "..")  # trsm/ paketini bulabilmek için (repo kökü)
from trsm.recommendation import compute_drug_statistics
from trsm.symptom_similarity import find_similar_phrases, load_symptom_index

DB_PATH = "../trsm.db"
INDEX_PATH = "../data/symptom_index.npz"
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.75
TOP_N_PER_SEED = 500
GUVEN_ESIGI = 90
KUCUK_ORNEKLEM_ESIGI = 20

conn = sqlite3.connect(DB_PATH)
model = SentenceTransformer(MODEL_NAME)
embeddings, ziyaret_ids, phrases = load_symptom_index(INDEX_PATH)

print(f"Semptom indeksi: {embeddings.shape[0]} kayıt, {embeddings.shape[1]} boyut")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Semptom indeksi: 96693 kayıt, 384 boyut


In [2]:
KATEGORILER = {
    "Uyku bozukluğu": ["uykusuzluk", "uyku düzensizliği", "aşırı uyku hali"],
    "Öfke / ajitasyon": ["öfke nöbeti", "ajitasyon", "saldırganlık", "huzursuzluk"],
    "Anksiyete / gerginlik": ["anksiyete", "gerginlik", "kaygı", "endişe hali"],
    "Depresif belirtiler": ["çökkünlük", "isteksizlik", "mutsuzluk", "enerji azlığı", "ağlama", "üzüntü"],
    "Manik / taşkın belirtiler": ["taşkınlık", "aşırı enerji", "hızlı konuşma", "elevasyon"],
    "Halüsinasyon / hezeyan": ["halüsinasyon", "sesler duyma", "hezeyan", "şüphecilik"],
    "İştah / kilo değişimi": ["iştahsızlık", "aşırı iştah", "kilo alımı", "kilo kaybı"],
    "Sosyal içe kapanma": ["içe kapanma", "sosyal izolasyon", "işlevsellik kaybı"],
    "Madde kullanımı": ["madde kullanımı", "alkol kullanımı", "bağımlılık"],
    "Tedaviye uyumsuzluk": ["ilaç bırakma", "düzensiz ilaç kullanımı", "tedaviyi reddetme"],
}

for kategori, tohumlar in KATEGORILER.items():
    print(f"{kategori}: {len(tohumlar)} tohum ifade")

Uyku bozukluğu: 3 tohum ifade
Öfke / ajitasyon: 4 tohum ifade
Anksiyete / gerginlik: 4 tohum ifade
Depresif belirtiler: 6 tohum ifade
Manik / taşkın belirtiler: 4 tohum ifade
Halüsinasyon / hezeyan: 4 tohum ifade
İştah / kilo değişimi: 4 tohum ifade
Sosyal içe kapanma: 3 tohum ifade
Madde kullanımı: 3 tohum ifade
Tedaviye uyumsuzluk: 3 tohum ifade


## Kategori Bazlı Ziyaret Eşleştirme

Her kategori için tohum ifadeler ayrı ayrı sorgulanır ve eşleşen
`ziyaret_id`'lerin birleşimi (union) alınır. Bu, tohum embedding'lerini
ortalamaktan kaçınmak için bilinçli bir tasarım tercihidir.

In [3]:
def kategori_esleyen_ziyaretler(tohumlar, embeddings, ziyaret_ids, phrases, model, min_similarity, top_n):
    """Her tohumu ayrı sorgu olarak çalıştırır, eşleşen ziyaret_id'lerin
    birleşimini döner. Tohum embedding'lerini ortalamak yerine ayrı ayrı
    sorgulamak bilinçli bir tercih: ortalama, konuşma diline yakın
    ifadelerin (örn. "mutsuzum ağlayasım geliyor") gerçek eşleşmesini
    sulandırıyor (bkz. tasarım dokümanı)."""
    eslenen = set()
    for tohum in tohumlar:
        qemb = model.encode(tohum)
        matches = find_similar_phrases(
            qemb, embeddings, ziyaret_ids, phrases,
            top_n=top_n, min_similarity=min_similarity,
        )
        eslenen.update(m["ziyaret_id"] for m in matches)
    return eslenen


kategori_ziyaretleri = {}
for kategori, tohumlar in KATEGORILER.items():
    kategori_ziyaretleri[kategori] = kategori_esleyen_ziyaretler(
        tohumlar, embeddings, ziyaret_ids, phrases, model,
        MIN_SIMILARITY, TOP_N_PER_SEED,
    )

kategori_ozet_df = pd.DataFrame([
    {
        "kategori": kategori,
        "eslesen_ziyaret_sayisi": len(ziyaret_id_seti),
        "kucuk_ornemklem": len(ziyaret_id_seti) < KUCUK_ORNEKLEM_ESIGI,
    }
    for kategori, ziyaret_id_seti in kategori_ziyaretleri.items()
]).sort_values("eslesen_ziyaret_sayisi", ascending=False)

kategori_ozet_df


,kategori,eslesen_ziyaret_sayisi,kucuk_ornemklem
1,Öfke / ajitasyon,993,False
3,Depresif belirtiler,967,False
9,Tedaviye uyumsuzluk,842,False
2,Anksiyete / gerginlik,757,False
0,Uyku bozukluğu,581,False
6,İştah / kilo değişimi,548,False
7,Sosyal içe kapanma,349,False
5,Halüsinasyon / hezeyan,249,False
8,Madde kullanımı,216,False
4,Manik / taşkın belirtiler,177,False
